In [ ]:
import pandas as pd
c = pd.read_csv("customers.csv", parse_dates=["signup_date"])
o = pd.read_csv("orders.csv", parse_dates=["order_date", "updated_at"])
a = pd.read_csv("customer_activity.csv", parse_dates=["event_date"])
for name, d in {"customers": c, "orders": o, "activity": a}.items():
    print(name, d.shape, "duplicate rows:", d.duplicated().sum(), "nulls:", d.isna().sum().sum())
print("unique order_id:", o.order_id.nunique(), "| orders without customer:", (~o.customer_id.isin(c.customer_id)).sum())
c.head()

In [ ]:
c["email_n"] = c.email.str.strip().str.lower()
c["phone_n"] = c.phone.str.replace(r"\D", "", regex=True)
c = c.sort_values("signup_date")
c["master_id"] = c.groupby("phone_n").customer_id.transform("first")
print(c.email.dropna().duplicated().sum(), c.email_n.dropna().duplicated().sum(), len(c) - c.master_id.nunique())
g = c.groupby("master_id")
conf = c[g.city.transform("nunique") > 1]
amb = c[c.groupby(["first_name", "last_name", "city"]).master_id.transform("nunique") > 1]
print(g.email_n.nunique().max(), conf.master_id.nunique(), amb.master_id.nunique())
conf.sort_values("master_id").head(4)

In [ ]:
total = o.amount.sum()
o = o.drop_duplicates()
n = len(o)
o = o.sort_values("updated_at").drop_duplicates("order_id", keep="last")
print(n, len(o), round(total - o.amount.sum(), 2))
o.status.value_counts()

In [ ]:
m = c.set_index("customer_id").master_id
o["master_id"] = o.customer_id.map(m)
a = a.drop_duplicates()
a["master_id"] = a.customer_id.map(m)
cust = c[c.customer_id == c.master_id].set_index("master_id")
j = o.join(cust.city, on="master_id", validate="m:1")
bad = o.merge(a, on="master_id")
print(len(o), len(j), len(bad), o.master_id.isna().sum(), a.master_id.isna().sum())

In [ ]:
early = a.event_date < a.master_id.map(cust.signup_date)
done = o[(o.status == "completed") & o.master_id.notna()]
f = done.groupby("master_id").agg(orders=("order_id", "size"), spent=("amount", "sum"), last_order=("order_date", "max"))
act = a[~early].groupby("master_id").agg(events=("event_type", "size"), last_seen=("event_date", "max"))
d = cust.join([f, act]).fillna({"orders": 0, "spent": 0, "events": 0})
d["email_n"] = g.email_n.first()
ref = pd.Timestamp("2025-12-31")
d["aov"] = (d.spent / d.orders).round(2)
d["recency_days"] = (ref - d.last_order).dt.days
d["inactive_90"] = ~((ref - d.last_seen).dt.days <= 90)
print("events before signup:", early.sum(), "| orders and events after cutoff:", (o.order_date > ref).sum(), (a.event_date > ref).sum())
d[["orders", "spent", "aov", "events", "recency_days"]].describe().round(1)

In [ ]:
print(d.index.is_unique, round(d.spent.sum(), 2), round(done.amount.sum(), 2))
print(round(o[o.master_id.isna()].amount.sum(), 2), d.inactive_90.sum(), (d.orders == 0).sum())

In [ ]:
x = pd.concat([conf[["customer_id", "master_id"]].assign(issue="city_conflict"),
               amb[["customer_id", "master_id"]].assign(issue="ambiguous_name"),
               o[o.master_id.isna()][["customer_id", "order_id"]].assign(issue="orphan_order"),
               a[a.master_id.isna()][["customer_id", "event_date"]].assign(issue="orphan_event"),
               a[early][["customer_id", "event_date"]].assign(issue="event_before_signup")])
x.to_csv("exceptions_report.csv", index=False)
d.to_csv("customers_consolidated.csv")
c.to_csv("customers.csv", index=False)
o.to_csv("orders.csv", index=False)
a.to_csv("customer_activity.csv", index=False)
x.issue.value_counts()

## Summary
- 542 records are 500 customers; 61 look-alikes kept apart.
- Duplicate orders inflated revenue by $11,580; totals match to the cent.
- $1,316 in 14 orders has no customer. Input files generated with AI.